## Data Preparation

## Imports & Config

In [1]:
import os, random
import numpy as np
import tensorflow as tf
import pandas as pd
import ast
import glob
import gc
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from PIL import Image
from keras_preprocessing.text import Tokenizer
from keras.utils import pad_sequences

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

BATCH_SIZE = 64
EPOCHS = 20
LEARNING_RATE= 0.001
MAX_LEN = 200
IMG_SIZE = 224
VOCAB_SIZE = 10000
EMBED_DIM = 128
DROPOUT_RATE = 0.3

DATA_DIR = '../data/raw'
PROCESSED_DIR = '../data/processed'
SPLITS_DIR = '../data/splits'

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(SPLITS_DIR, exist_ok=True)

## Load Data

In [2]:
dataset_path = os.path.join(DATA_DIR, 'dataset.csv')
steam_path = os.path.join(DATA_DIR, 'steam.csv')

df_reviews = pd.read_csv(dataset_path, nrows=100000)
df_steam = pd.read_csv(steam_path)


In [3]:
# Filter to games where we have images
images_list = glob.glob(os.path.join(DATA_DIR, 'images', '*.jpg'))
downloaded_app_ids = [int(os.path.basename(p).split('.')[0]) for p in images_list]

print(f"Total downloaded images: {len(downloaded_app_ids)}")

df_reviews = df_reviews[df_reviews['app_id'].isin(downloaded_app_ids)]
print(f"Reviews for downloaded games: {len(df_reviews)}")

# Sample to 15000 rows
if len(df_reviews) > 15000:
    df_reviews = df_reviews.sample(n=15000, random_state=SEED)
df_reviews = df_reviews.reset_index(drop=True)
print(f"Sampled reviews: {len(df_reviews)}")

Total downloaded images: 15000
Reviews for downloaded games: 92117
Sampled reviews: 15000


## Data Preparation

### Text Prep

In [4]:
# Merge metadata
df_merged = df_reviews.merge(df_steam, left_on='app_id', right_on='appid', how='left')

reviews = df_merged['review_text'].astype(str).fillna('').values

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(reviews)
sequences = tokenizer.texts_to_sequences(reviews)
X_text = pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')

np.save(os.path.join(PROCESSED_DIR, 'X_text.npy'), X_text)

import pickle
with open(os.path.join(PROCESSED_DIR, 'tokenizer.pkl'), 'wb') as f:
    pickle.dump(tokenizer, f)


### Image Prep

In [ ]:
# Image Prep
def load_and_preprocess_image(app_id):
    path = os.path.join(DATA_DIR, 'images', f"{app_id}.jpg")
    try:
        img = Image.open(path).convert('RGB')
        img = img.resize((IMG_SIZE, IMG_SIZE))
        # Cast to float16 to save memory (uses 2 bytes instead of 8 bytes per pixel)
        img_array = np.array(img, dtype=np.float16) / 255.0
        return img_array
    except:
        return np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.float16)

import gc
# Pre-allocate the array to avoid memory spikes from list comprehension
if os.path.exists(os.path.join(PROCESSED_DIR, 'X_img.npy')):
    print('X_img.npy already exists')
else:
    X_img = np.zeros((len(df_merged), IMG_SIZE, IMG_SIZE, 3), dtype=np.float16)

    for i, appid in enumerate(df_merged['app_id']):
            X_img[i] = load_and_preprocess_image(appid)

    np.save(os.path.join(PROCESSED_DIR, 'X_img.npy'), X_img)
gc.collect()

20

### Features Prep

In [ ]:
df_merged['review_text'] = df_merged['review_text'].fillna('').astype(object)

df_merged['review_score_norm'] = df_merged['review_score'].fillna(0) / 10.0
df_merged['playtime_norm'] = np.log1p(df_merged['average_playtime'].fillna(0))
df_merged['word_count'] = df_merged['review_text'].str.split().str.len().fillna(0).astype(int)
df_merged['is_repetitive'] = (df_merged['word_count'] < 5).astype(int)
df_merged['price_norm'] = df_merged['price'].fillna(0) / 100.0
df_merged['review_count_log'] = np.log1p(df_merged['review_votes'].fillna(0))
df_merged['positive_ratio'] = (df_merged['positive_ratings'].fillna(0) / (df_merged['positive_ratings'].fillna(0) + df_merged['negative_ratings'].fillna(1)))
df_merged['has_achievements'] = (df_merged['achievements'].fillna(0) > 0).astype(int)
df_merged['median_playtime_norm'] = np.log1p(df_merged['median_playtime'].fillna(0))
df_merged['required_age_norm'] = df_merged['required_age'].fillna(0) / 18.0

features = ['review_score_norm','playtime_norm','word_count','is_repetitive','price_norm',
            'review_count_log','positive_ratio','has_achievements','median_playtime_norm','required_age_norm']
            
X_features = df_merged[features].fillna(0).values.astype('float32')
np.save(os.path.join(PROCESSED_DIR, 'X_features.npy'), X_features)
print(f"X_features shape: {X_features.shape}")


Saved X_features.npy with shape: (15000, 10)


## Label Creation

In [7]:
# sentiment_label (1 for positive, 0 for negative)
y_sentiment = df_merged['review_score'].apply(lambda x: 1 if x > 0 else 0).values

# fake_label (is_repetitive OR word_count<10)
y_fake = ((df_merged['is_repetitive'] == 1) | (df_merged['word_count'] < 10)).astype(int).values

# match_label (genre logic)
GENRE_KEYWORDS = {
    'Action': ['shoot','fight','combat','kill','weapon','battle','gun','fps','enemy'],
    'Adventure': ['explore','quest','story','journey','world','discover','puzzle','mystery'],
    'RPG': ['level','build','character','boss','skill','upgrade','dialogue','lore'],
    'Strategy': ['build','resource','manage','plan','turn','base','army','economy'],
    'Simulation': ['simulate','build','manage','city','farm','drive','craft','sandbox'],
    'Horror': ['scary','fear','dark','jumpscare','tense','atmosphere','survive','monster'],
    'Sports': ['team','score','match','season','league','player','race','tournament'],
    'Casual': ['fun','easy','relax','simple','cute','colorful','chill','light'],
    'Indie': ['creative','unique','artistic','pixel','retro','atmospheric','story'],
    'Racing': ['race','car','speed','drift','track','lap','vehicle','drive'],
}

def get_primary_genre(genres_str):
    if pd.isna(genres_str): return 'Unknown'
    return str(genres_str).split(';')[0].strip()

def make_match_label(row):
    genre = get_primary_genre(row.get('genres', ''))
    if genre not in GENRE_KEYWORDS:
        return 0
    review_words = set(str(row.get('review_text', '')).lower().split())
    keywords = set(GENRE_KEYWORDS[genre])
    return 1 if len(review_words & keywords) > 0 else 0

y_match = df_merged.apply(make_match_label, axis=1).values.astype(int)
print("match_label distribution:", pd.Series(y_match).value_counts().to_dict())

np.save(os.path.join(PROCESSED_DIR, 'y_sentiment.npy'), y_sentiment)
np.save(os.path.join(PROCESSED_DIR, 'y_fake.npy'), y_fake)
np.save(os.path.join(PROCESSED_DIR, 'y_match.npy'), y_match)


## Train/Val/Test Split

In [8]:
indices = np.arange(len(df_merged))
idx_train, idx_temp = train_test_split(indices, test_size=0.3, random_state=SEED)
idx_val, idx_test = train_test_split(idx_temp, test_size=0.5, random_state=SEED)

np.save(os.path.join(SPLITS_DIR, 'idx_train.npy'), idx_train)
np.save(os.path.join(SPLITS_DIR, 'idx_val.npy'), idx_val)
np.save(os.path.join(SPLITS_DIR, 'idx_test.npy'), idx_test)

print(f"Train: {len(idx_train)}, Val: {len(idx_val)}, Test: {len(idx_test)}")


Train: 10500, Val: 2250, Test: 2250
